# 📖 Notebook 7: GitOps with ArgoCD — Deploying from Git

Welcome! In this notebook, you will learn a deployment style called **GitOps**. The big idea is simple: instead of clicking buttons or running lots of manual commands every time you want to deploy, you store the desired Kubernetes state in Git and let a tool keep the cluster matched to that Git history.

We will use the sample microservices in the `k8s-lab` namespace:
- `api-gateway` on port `8000`
- `user-service` on port `8001`
- `order-service` on port `8002`

Think of Git as the **blueprint drawer** and ArgoCD as the **worker that keeps rebuilding the real system to match the blueprint**.


## Learning Objectives

By the end of this notebook, you will be able to:

- explain the main GitOps idea in beginner-friendly words
- describe why Git becomes the single source of truth
- install ArgoCD into a Kubernetes cluster
- create an ArgoCD Application for the `k8s-lab` manifests
- observe sync, auto-sync, and self-healing behavior
- understand how Git commits and Git revert can act like deploy and rollback buttons


## 🛠️ Setup

Before starting:

1. Make sure your Kubernetes cluster is running.
2. Make sure the `k8s-lab` namespace and sample services already exist.
3. Make sure Git is installed on your machine.
4. Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → 'Reload Window'.

This notebook uses shell commands inside code cells, so you can follow the flow step by step.


In [ ]:
!kubectl cluster-info
!kubectl get ns
!kubectl get deployments -n k8s-lab
!git --version


## What is GitOps?

GitOps means **Git describes what the cluster should look like**. A tool such as ArgoCD keeps checking Git and comparing it with the real cluster. If the cluster drifts away from the files in Git, ArgoCD can bring it back.

A good beginner mental model is:

- **Git** = your saved plan
- **Kubernetes** = the real running system
- **ArgoCD** = the careful robot inspector

### 🧪 Practical Exercise
Look at the existing manifests in this lab and say out loud what they describe: namespaces, deployments, services, and other cluster objects. The goal is to notice that GitOps works best when your desired state is already written down as files.


In [ ]:
!ls -1 manifests
!kubectl get all -n k8s-lab


## 📦 Git Push → ArgoCD → Kubernetes

```text
+-------------------+        +-------------------+        +------------------------+
|   Your Git Repo   | -----> |      ArgoCD       | -----> |   Kubernetes Cluster   |
| manifests/*.yaml  |        | watches Git state |        | runs real workloads    |
+-------------------+        +-------------------+        +------------------------+
         ^                              |                               |
         |                              v                               v
         +---------------------- Git is the source ---------------------+
```

ArgoCD keeps asking one question: **"Does the cluster still match Git?"**


## Traditional Deploy vs GitOps

| Topic | Traditional Deploy | GitOps |
|---|---|---|
| Source of truth | Terminal history, wiki pages, memory | Git repository |
| Change process | Humans run commands by hand | Humans change Git, ArgoCD applies it |
| Audit trail | Harder to reconstruct | Built into Git commits |
| Rollback | Manual and sometimes stressful | Revert a commit and sync again |
| Drift recovery | Someone must notice and fix it | ArgoCD can self-heal |
| Team collaboration | Easy to make different changes in different ways | Everyone works through the same repo |

### 🧪 Practical Exercise
Pick one row from the table and explain why it matters to a small team. For example, why is an audit trail useful when several people deploy the same app?


## 1) Install ArgoCD

First, we install ArgoCD into its own namespace. This adds controllers, APIs, and a web UI.

### 🧪 Practical Exercise
Before running the install, guess what will happen after we create the `argocd` namespace: will you see more pods, more services, or both? Then run the cell and check your answer.


In [ ]:
!kubectl create namespace argocd
!kubectl apply -n argocd -f https://raw.githubusercontent.com/argoproj/argo-cd/stable/manifests/install.yaml
!kubectl wait --for=condition=ready pod -l app.kubernetes.io/name=argocd-server -n argocd --timeout=120s
!kubectl get pods -n argocd


## 2) Access the ArgoCD UI

The ArgoCD server runs inside the cluster. To open it in your browser from your laptop, we use **port-forwarding**. That creates a temporary tunnel from your machine to the ArgoCD service.

Then we fetch the initial admin password from a Kubernetes secret.

### 🧪 Practical Exercise
After starting the port-forward, open `https://localhost:8080` in your browser. Log in with username `admin` and the password from the next command. Notice how the UI starts empty before any Application is created.


In [ ]:
!kubectl port-forward svc/argocd-server -n argocd 8080:443 &
!kubectl -n argocd get secret argocd-initial-admin-secret -o jsonpath="{.data.password}" | base64 -d && echo
!echo 'macOS with Homebrew: brew install argocd'
!echo 'Without Homebrew, follow the ArgoCD CLI install guide from the official docs.'


## 3) Create a local Git repo for GitOps

Now we create a small Git repository that contains the Kubernetes manifests. This helps you see the Git side of GitOps clearly. To keep the lab self-contained, we will place it in `./gitops-workdir/k8s-lab-gitops`.

> Important beginner note: ArgoCD cannot watch a random folder on your laptop directly. It watches a Git repository that **it can reach**. So we will create a local repo first because it is easy to understand, and then point ArgoCD at a remote Git URL that you push to.

### 🧪 Practical Exercise
After the commit finishes, run `git log --oneline` mentally or in the terminal and notice that your desired cluster state now has a version number: a commit hash. That is one of the biggest GitOps wins.


In [ ]:
!rm -rf ./gitops-workdir/k8s-lab-gitops && mkdir -p ./gitops-workdir/k8s-lab-gitops && cp -R manifests ./gitops-workdir/k8s-lab-gitops/ && cd ./gitops-workdir/k8s-lab-gitops && git init -b main && git config user.name "K8s Lab" && git config user.email "lab@example.com" && git add . && git commit -m "Initial GitOps manifests"
!cd ./gitops-workdir/k8s-lab-gitops && git log --oneline -n 3


## 4) Create an ArgoCD Application

An **Application** is the object that tells ArgoCD:

- which repo to watch
- which folder inside that repo to read
- which cluster and namespace to deploy into

For this lab, the source is the `manifests` directory in a Git repo. Replace `YOUR_USERNAME` with a real GitHub username after you push your local repo to GitHub or another Git server that ArgoCD can reach.

### 🧪 Practical Exercise
Read the YAML before applying it and answer these two questions: (1) What namespace will ArgoCD deploy into? (2) What two automation features are turned on?


In [ ]:
!printf '%s\n' 'apiVersion: argoproj.io/v1alpha1' 'kind: Application' 'metadata:' '  name: k8s-lab-app' '  namespace: argocd' 'spec:' '  project: default' '  source:' '    repoURL: https://github.com/YOUR_USERNAME/k8s-lab-gitops.git' '    targetRevision: main' '    path: manifests' '    directory:' '      recurse: true' '  destination:' '    server: https://kubernetes.default.svc' '    namespace: k8s-lab' '  syncPolicy:' '    automated:' '      prune: true' '      selfHeal: true' '    syncOptions:' '      - CreateNamespace=true' > manifests/argocd-app.yaml
!cat manifests/argocd-app.yaml
!kubectl apply -f manifests/argocd-app.yaml
!kubectl get applications -n argocd


## 5) Observe sync and self-healing

Once the Application exists, ArgoCD starts comparing Git with the cluster. If something in the cluster disappears but Git still says it should exist, ArgoCD can recreate it. That is called **self-healing**.

### 🧪 Practical Exercise
Delete one deployment by hand and predict what ArgoCD will do next. Will it ignore the change, complain only in the UI, or recreate the missing deployment?


In [ ]:
!kubectl get applications -n argocd
!kubectl delete deployment user-service -n k8s-lab
!sleep 15 && kubectl get deployments -n k8s-lab


## 6) Make a change in Git and watch auto-sync

A very common GitOps flow is:

1. edit the manifest
2. commit the change
3. push the change
4. let ArgoCD notice and sync it

Below, we change the `api-gateway` replica count from `2` to `3`.

### 🧪 Practical Exercise
Before you run the cell, guess where you will see the new desired state first: Git history, ArgoCD UI, or the Kubernetes Deployment object. Then run the commands and compare the order.


In [ ]:
!echo 'Replace YOUR_USERNAME before running the push commands for real.'
!cd ./gitops-workdir/k8s-lab-gitops && (git remote remove origin 2>/dev/null || true) && git remote add origin https://github.com/YOUR_USERNAME/k8s-lab-gitops.git && git push -u origin main
!cd ./gitops-workdir/k8s-lab-gitops && python3 -c "from pathlib import Path; p = Path('manifests/deployment.yaml'); p.write_text(p.read_text().replace('replicas: 2', 'replicas: 3', 1))" && git add manifests/deployment.yaml && git commit -m "Scale api-gateway to 3 replicas" && git push
!sleep 15 && kubectl get deployment api-gateway -n k8s-lab


## 7) Roll back with Git revert

Rollback in GitOps is beautifully simple: if a bad change entered the repo, undo the commit and let the system converge back to the older good state.

### 🧪 Practical Exercise
Run the revert and then explain why this rollback is easier to review than running random emergency commands straight in the cluster.


In [ ]:
!cd ./gitops-workdir/k8s-lab-gitops && git revert --no-edit HEAD && git push
!sleep 15 && kubectl get deployment api-gateway -n k8s-lab


## 🧹 Clean Up

When you finish exploring, remove ArgoCD from the cluster so the lab returns to a simpler state.

### 🧪 Practical Exercise
After deleting the namespace, run `kubectl get ns` and verify that `argocd` disappears. This helps reinforce that most Kubernetes tools are just resources living in namespaces.


In [ ]:
!kubectl delete namespace argocd


## 🎓 What You Learned

Nice work. In this notebook, you learned that:

- GitOps uses Git as the single source of truth
- ArgoCD watches Git and compares it to the live cluster
- auto-sync lets changes flow from Git to Kubernetes automatically
- self-heal lets ArgoCD repair drift when someone changes the cluster manually
- Git commits and Git revert create a clean, reviewable deployment history

If you can explain the difference between **changing the cluster directly** and **changing Git first**, you now understand the heart of GitOps.
